In [38]:
import kagglehub

path = kagglehub.dataset_download("sriramr/fruits-fresh-and-rotten-for-classification")

Using Colab cache for faster access to the 'fruits-fresh-and-rotten-for-classification' dataset.


In [39]:
import tensorflow as tf
import os
train_path = os.path.join(path,'dataset','train')

In [40]:
img_height = 224
img_width = 224
batch_size = 32

In [41]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset='training',
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset='validation',
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)


Found 10901 files belonging to 6 classes.
Using 8721 files for training.
Found 10901 files belonging to 6 classes.
Using 2180 files for validation.


In [42]:
original_class_name_for_display = train_ds.class_names
print(f"Original Class Name for Display: {original_class_name_for_display}")
print(f"Number of batches in training dataset: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Number of batches in validation dataset: {tf.data.experimental.cardinality(val_ds).numpy()}")


Original Class Name for Display: ['freshapples', 'freshbanana', 'freshoranges', 'rottenapples', 'rottenbanana', 'rottenoranges']
Number of batches in training dataset: 273
Number of batches in validation dataset: 69


In [43]:
def normalize(image,lable):
  image = tf.cast(image/255. ,tf.float32)
  return image, lable

In [44]:
train_ds = train_ds.map(normalize)
val_ds = val_ds.map(normalize)

In [45]:
def map_binary(image, label):
  binary_lable = tf.cast(label >= len(original_class_name_for_display)//2, tf.float32)
  return image, binary_lable

In [46]:
train_ds = train_ds.map(map_binary)
val_ds = val_ds.map(map_binary)

In [47]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2)
])

def augment(image, label):
  image = data_augmentation(image)
  return image, label

train_ds_augmented = train_ds.map(augment)

In [48]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(img_height, img_width, 3)),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [49]:
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [50]:
model.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

In [51]:
epochs = 10

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_ds_augmented,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=[early_stopping]
)

Epoch 1/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 131s 464ms/step - accuracy: 0.8602 - loss: 0.3310 - val_accuracy: 0.9317 - val_loss: 0.1831
Epoch 2/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 121s 441ms/step - accuracy: 0.9140 - loss: 0.2157 - val_accuracy: 0.8922 - val_loss: 0.2392
Epoch 3/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 122s 447ms/step - accuracy: 0.9287 - loss: 0.1798 - val_accuracy: 0.9358 - val_loss: 0.1542
Epoch 4/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 122s 445ms/step - accuracy: 0.9326 - loss: 0.1706 - val_accuracy: 0.9381 - val_loss: 0.1533
Epoch 5/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 122s 445ms/step - accuracy: 0.9392 - loss: 0.1464 - val_accuracy: 0.9509 - val_loss: 0.1152
Epoch 6/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 122s 447ms/step - accuracy: 0.9448 - loss: 0.1388 - val_accuracy: 0.9463 - val_loss: 0.1255
Epoch 7/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 121s 442ms/step - accuracy: 0.9485 - loss: 0.1283 - val_accuracy: 0.9560 - val_loss: 0.0973
Epoch 8/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 120s 437ms/step - accuracy: 0.9426 -

In [56]:
test_path = os.path.join(path, 'dataset', 'test')

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_path,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle = False
)

Found 2698 files belonging to 6 classes.


In [57]:
test_ds = test_ds.map(normalize)
test_ds = test_ds.map(map_binary)

In [58]:
loss, accuracy = model.evaluate(test_ds)
test_loss = loss
test_accuracy = accuracy

print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

85/85 ━━━━━━━━━━━━━━━━━━━━ 9s 102ms/step - accuracy: 0.9655 - loss: 0.0879
Test Loss: 0.08793362230062485
Test Accuracy: 0.9655300378799438


In [59]:
model_save_path = 'fruits_classification_model.h5'
model.save(model_save_path)

In [60]:
model_save_path = 'fruits_classification_model.keras'
model.save(model_save_path)